# 01 — Exploração dos Dados

Este notebook apresenta a exploração inicial do conjunto de treinamento utilizado no projeto de classificação de clareza.

O objetivo é caracterizar o corpus antes da modelagem, observando:

- quantidade de exemplos;
- distribuição das classes;
- valores ausentes e textos vazios;
- tamanho dos documentos;
- possíveis diferenças de comprimento entre as classes.

Nenhum modelo é treinado aqui. A modelagem e os baselines são tratados nos experimentos posteriores.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Permite executar o notebook tanto a partir da pasta notebooks/
# quanto a partir da raiz do repositório.
CWD = Path.cwd().resolve()

if (CWD / "src").exists():
    ROOT = CWD
elif (CWD.parent / "src").exists():
    ROOT = CWD.parent
else:
    raise RuntimeError(
        "Não foi possível localizar a raiz do projeto. "
        "Execute o notebook dentro do repositório ep1-pln."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import CLASSES
from src.data import carregar_dados_treino


In [ ]:
X, y = carregar_dados_treino()

dados = pd.DataFrame({
    "resp_text": X,
    "clarity": y,
})

print("Quantidade de exemplos:", len(dados))
print("Colunas:", list(dados.columns))
print("Classes:", CLASSES)

dados.head()


## Distribuição das classes

A distribuição das classes é importante porque a acurácia pode ser enganosa quando existe forte desbalanceamento.

Além da acurácia, o projeto registra também Macro-F1, que calcula o F1 de cada classe separadamente antes de fazer a média.


In [ ]:
distribuicao = (
    dados["clarity"]
    .value_counts()
    .reindex(CLASSES)
    .rename("quantidade")
    .to_frame()
)

distribuicao["proporcao"] = (
    distribuicao["quantidade"]
    / len(dados)
)

distribuicao


In [ ]:
ax = (
    distribuicao["quantidade"]
    .plot(
        kind="bar",
        figsize=(7, 4),
        title="Distribuição das classes",
    )
)

ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade de exemplos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Valores ausentes e textos vazios


In [ ]:
print(
    "Valores ausentes em resp_text:",
    dados["resp_text"].isna().sum(),
)

print(
    "Valores ausentes em clarity:",
    dados["clarity"].isna().sum(),
)

textos_vazios = (
    dados["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Textos vazios:", textos_vazios)


## Tamanho dos documentos

Como complemento à distribuição das classes, analisamos o comprimento das respostas em caracteres e em número aproximado de palavras.


In [ ]:
dados_exploracao = dados.copy()

dados_exploracao["n_caracteres"] = (
    dados_exploracao["resp_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

dados_exploracao["n_palavras"] = (
    dados_exploracao["resp_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

dados_exploracao[
    ["n_caracteres", "n_palavras"]
].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)


In [ ]:
plt.figure(figsize=(8, 4))

dados_exploracao["n_palavras"].hist(
    bins=50,
)

plt.title("Distribuição do tamanho dos textos")
plt.xlabel("Número aproximado de palavras")
plt.ylabel("Quantidade de documentos")
plt.tight_layout()
plt.show()


In [ ]:
analise_por_classe = (
    dados_exploracao
    .groupby("clarity")
    .agg(
        quantidade=("clarity", "size"),
        media_palavras=("n_palavras", "mean"),
        mediana_palavras=("n_palavras", "median"),
        media_caracteres=("n_caracteres", "mean"),
        mediana_caracteres=("n_caracteres", "median"),
    )
    .reindex(CLASSES)
)

analise_por_classe


## Conclusões da exploração

A exploração fornece a caracterização básica do conjunto de treinamento utilizada nos experimentos seguintes.

Os pontos observados aqui servem principalmente para:

- verificar a distribuição das três classes;
- confirmar a qualidade básica das entradas;
- identificar documentos vazios ou muito curtos;
- entender a variação de comprimento das respostas;
- justificar o acompanhamento conjunto de Accuracy e Macro-F1.

A comparação de modelos é feita separadamente, utilizando validação estratificada e os procedimentos definidos no código do projeto.
